<a href="https://colab.research.google.com/github/hackBytesLab/ai-financial-analyst/blob/main/ComfyUI_T4_Krea2_fixed_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ComfyUI T4 — Krea 2 (เวอร์ชันแก้ไข)

ใช้ Krea 2 บน Tesla T4 โดยเปิดหน้า ComfyUI ผ่าน **Cloudflare tunnel** (แทน iframe ของ Colab ที่เปิดไม่สำเร็จ)

**ขั้นตอน**
1. เลือก **Runtime → Change runtime type → T4 GPU**
2. กด **Run all** หรือรันเซลล์ 1–6 ตามลำดับ (ไม่ต้องรีสตาร์ท ComfyUI เองอีกแล้ว เพราะโมเดลถูกติดตั้งก่อนเริ่มเซิร์ฟเวอร์)
   - เซลล์ 2 อนุญาตเชื่อม Google Drive
   - เซลล์ 4 คัดลอกโมเดลจาก Drive + ดาวน์โหลด text encoder / VAE และตรวจ checksum
   - เซลล์ 5 เริ่ม ComfyUI และตรวจว่าเห็นโมเดล, encoder, VAE, blueprint
   - เซลล์ 6 เปิด tunnel แล้วพิมพ์ลิงก์ `trycloudflare.com` ให้คลิก
3. ใช้ blueprint **Text to Image (Krea-2 Turbo)** ในหน้า ComfyUI
4. เสร็จแล้วให้ **Runtime → Disconnect and delete runtime** เพื่อหยุดทุกอย่างและหยุดนับเวลา GPU

**ที่เก็บไฟล์**
- โมเดลหลัก: `MyDrive/ComfyUI_T4_Clothing/models/`
- ภาพที่สร้าง: `MyDrive/ComfyUI_T4_Clothing/results/` (ComfyUI เขียนลง Drive โดยตรง)
- encoder / VAE / ComfyUI อยู่ใน runtime ของ Colab และหายเมื่อ runtime รีเซ็ต (ดาวน์โหลดใหม่อัตโนมัติ)

**ข้อควรระวัง**
- ลิงก์ tunnel เป็นสาธารณะ ใครมี URL ก็เข้าได้ขณะที่ยังรันอยู่ อย่าแชร์ลิงก์ และหยุดเมื่อเสร็จ
- ขอบเขต: โน้ตบุ๊กนี้รองรับ workflow ทางการของ Krea 2 (สร้างภาพ / อ้างอิงสไตล์) ไม่รวม workflow แทนที่เสื้อผ้าด้วย mask แบบเก่า

In [ ]:
# 1. T4 only: fail before downloading or installing anything.
import os, sys, subprocess, json, time, pathlib, hashlib, shutil, tarfile, zipfile, urllib.request, re, struct
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Select Runtime > Change runtime type > T4 GPU.")
gpu = torch.cuda.get_device_name(0)
if gpu.strip() not in ("Tesla T4", "NVIDIA T4"):
    raise RuntimeError(f"T4 required; received {gpu}. No installation was started.")
print("GPU:", gpu, "VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
print("Python:", sys.version.split()[0], "Torch:", torch.__version__)

GPU: Tesla T4 VRAM: 14.6 GiB
Python: 3.13.15 Torch: 2.11.0+cu130


In [ ]:
# 2. Google Drive: model source, dependency cache and results.
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)
BASE = pathlib.Path('/content/drive/MyDrive/ComfyUI_T4_Clothing')
CACHE = BASE / 'cache'
RESULTS = BASE / 'results'
MODELS = BASE / 'models'
for p in (CACHE, RESULTS):
    p.mkdir(parents=True, exist_ok=True)
print('Cache:  ', CACHE)
print('Results:', RESULTS)
print('Models: ', MODELS, '(exists)' if MODELS.is_dir() else '(MISSING - create it and upload the model)')

Mounted at /content/drive
Cache:   /content/drive/MyDrive/ComfyUI_T4_Clothing/cache
Results: /content/drive/MyDrive/ComfyUI_T4_Clothing/results
Models:  /content/drive/MyDrive/ComfyUI_T4_Clothing/models (exists)


In [ ]:
# 3. Install ComfyUI + dependencies (cached on Drive). No models are downloaded here.
import importlib.metadata, importlib.util, platform

# Pinned commit for reproducibility. If cell 5 reports that the Krea 2 blueprint is missing,
# this commit is too old: set COMFY_REF = "master" (a branch name is always re-downloaded fresh).
COMFY_REF = "30c4c3aa756f4655cde89c287c0745381a6c2bbc"

APP = pathlib.Path('/content/ComfyUI_T4')
ENV = pathlib.Path('/content/comfy-t4-venv')
PY = str(ENV / 'bin/python')

def checked(args, **kw):
    result = subprocess.run([str(x) for x in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kw)
    if result.returncode:
        raise RuntimeError(result.stdout[-10000:])
    if result.stdout:
        print(result.stdout[-2000:])

def fetch(url, dest):
    dest = pathlib.Path(dest)
    if dest.exists():
        return dest
    print('Downloading:', dest.name, flush=True)
    temp = dest.with_name(dest.name + '.partial')
    request = urllib.request.Request(url, headers={'User-Agent': 'ComfyUI-T4-notebook'})
    with urllib.request.urlopen(request, timeout=120) as response, temp.open('wb') as f:
        received, next_report = 0, 256 * 1024 * 1024
        while True:
            block = response.read(8 * 1024 * 1024)
            if not block:
                break
            f.write(block)
            received += len(block)
            if received >= next_report:
                print(round(received / 2**30, 2), 'GiB downloaded', flush=True)
                next_report += 256 * 1024 * 1024
    os.replace(temp, dest)
    return dest

zip_path = CACHE / f'comfy-{COMFY_REF}.zip'
if not re.fullmatch(r'[0-9a-f]{40}', COMFY_REF):
    zip_path.unlink(missing_ok=True)   # branch names move: always fetch fresh
source = fetch(f'https://github.com/Comfy-Org/ComfyUI/archive/{COMFY_REF}.zip', zip_path)

if not (APP / 'main.py').exists():
    with zipfile.ZipFile(source) as z:
        root = z.namelist()[0].split('/')[0] + '/'
        for item in z.infolist():
            rel = pathlib.PurePosixPath(item.filename).relative_to(root)
            if '..' in rel.parts:
                raise RuntimeError('Unsafe source archive path')
            dest = APP.joinpath(*rel.parts)
            if item.is_dir():
                dest.mkdir(parents=True, exist_ok=True)
            else:
                dest.parent.mkdir(parents=True, exist_ok=True)
                with z.open(item) as src, dest.open('wb') as out:
                    shutil.copyfileobj(src, out)

requirements = (APP / 'requirements.txt').read_bytes()
base = sorted((d.metadata['Name'].lower(), d.version) for d in importlib.metadata.distributions() if d.metadata.get('Name'))
signature = hashlib.sha256(json.dumps([COMFY_REF, sys.version, platform.machine(), base, requirements.decode()], sort_keys=True).encode()).hexdigest()[:20]
archive = CACHE / f'environment-{signature}.tar.gz'
stamp = ENV / '.ready'

if not stamp.exists() or stamp.read_text() != signature:
    if archive.exists():
        print('Restoring cached dependencies…')
        with tarfile.open(archive) as t:
            t.extractall('/content', filter='data')
    else:
        print('First setup: installing dependencies (keeping Colab CUDA/PyTorch).')
        checked([sys.executable, '-m', 'venv', '--without-pip', '--copies', '--system-site-packages', ENV])
        constraints = APP / 'runtime-constraints.txt'
        constraints.write_text('\n'.join(f'{name}=={importlib.metadata.version(name)}' for name in ('torch', 'torchvision') if importlib.util.find_spec(name)))
        checked([PY, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', APP / 'requirements.txt', '-c', constraints])
        stamp.write_text(signature)
        temporary = archive.with_name(archive.name + '.partial')
        with tarfile.open(temporary, 'w:gz', compresslevel=1) as t:
            t.add(ENV, arcname=ENV.name)
        os.replace(temporary, archive)
    if not stamp.exists() or stamp.read_text() != signature:
        raise RuntimeError('Dependency cache mismatch')

print('ComfyUI source:', COMFY_REF[:12], '| dependency cache hit:', archive.exists())
print('Install complete. Next: cell 4 (models).')

Restoring cached dependencies…
ComfyUI source: 30c4c3aa756f | dependency cache hit: True
Install complete. Next: cell 4 (models).


In [ ]:
# 4. Krea 2 assets: copy the Drive model, download encoder + VAE, verify checksums.
MODEL_NAME = 'moodyKrea2MixUncensoredV8SEGoes_v80_int4.safetensors'
SRC_MODEL = MODELS / MODEL_NAME
if not SRC_MODEL.is_file():
    raise FileNotFoundError(f'Model not found in Drive: {SRC_MODEL}. Confirm Drive is mounted and the file is inside MyDrive/ComfyUI_T4_Clothing/models/.')

# --- Inspect the model header (cheap) to see what format the int4 file really is. ---
def inspect_safetensors(path):
    with open(path, 'rb') as f:
        n = struct.unpack('<Q', f.read(8))[0]
        if n > 200 * 1024 * 1024:
            raise RuntimeError('Not a valid safetensors header')
        header = json.loads(f.read(n))
    meta = header.pop('__metadata__', {}) or {}
    dtypes = {}
    for v in header.values():
        dtypes[v['dtype']] = dtypes.get(v['dtype'], 0) + 1
    keys = list(header)
    markers = ('qweight', 'wscales', 'wcscales', 'wtscale', 'smooth_factor', 'proj_down', 'proj_up')
    nunchaku = any(m in k for k in keys for m in markers)
    return len(keys), dtypes, nunchaku, meta, keys[:3]

count, dtypes, nunchaku_like, meta, sample = inspect_safetensors(SRC_MODEL)
print(f'Model header: {count} tensors, dtypes={dtypes}')
print('Sample keys:', sample)
if meta:
    print('Metadata keys:', list(meta)[:10])
if nunchaku_like:
    print('WARNING: tensor names look like Nunchaku/SVDQuant int4. The standard UNETLoader cannot load that format.')
    print('         You would need the matching custom node; then set DISABLE_CUSTOM_NODES = False in cell 5.')

# --- Copy model to the runtime SSD. ---
model_dir = APP / 'models' / 'diffusion_models'
text_dir = APP / 'models' / 'text_encoders'
vae_dir = APP / 'models' / 'vae'
for folder in (model_dir, text_dir, vae_dir):
    folder.mkdir(parents=True, exist_ok=True)

model_target = model_dir / MODEL_NAME
if not model_target.exists() or model_target.stat().st_size != SRC_MODEL.stat().st_size:
    print('Copying model from Drive…', flush=True)
    temp = model_target.with_name(model_target.name + '.partial')
    shutil.copyfile(SRC_MODEL, temp)
    os.replace(temp, model_target)
print(f'Model ready: {model_target.name} ({model_target.stat().st_size / 2**30:.2f} GiB)')

# --- Official encoder + VAE, verified against the Hugging Face LFS SHA-256. ---
HF_REPO = 'Comfy-Org/Krea-2'
ASSETS = [
    ('text_encoders/qwen3vl_4b_fp8_scaled.safetensors', text_dir / 'qwen3vl_4b_fp8_scaled.safetensors'),
    ('vae/qwen_image_vae.safetensors', vae_dir / 'qwen_image_vae.safetensors'),
]
try:
    listing = json.load(urllib.request.urlopen(f'https://huggingface.co/api/models/{HF_REPO}?blobs=true', timeout=60))
    expected = {s['rfilename']: s.get('lfs') for s in listing['siblings']}
except Exception as e:
    expected = {}
    print('WARNING: could not fetch checksums from Hugging Face; falling back to a size check only:', e)

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def asset_ok(path, lfs):
    if not path.exists():
        return False
    if lfs:
        return path.stat().st_size == lfs['size'] and sha256_of(path) == lfs['sha256']
    return path.stat().st_size > 1024 * 1024

for remote, dest in ASSETS:
    lfs = expected.get(remote)
    if asset_ok(dest, lfs):
        print(f'Already present and verified: {dest.name} ({dest.stat().st_size / 2**30:.2f} GiB)')
        continue
    dest.unlink(missing_ok=True)
    fetch(f'https://huggingface.co/{HF_REPO}/resolve/main/{remote}', dest)
    if not asset_ok(dest, lfs):
        dest.unlink(missing_ok=True)
        raise RuntimeError(f'Checksum/size check failed for {dest.name}; file removed. Re-run this cell.')
    print(f'Ready: {dest.name} ({dest.stat().st_size / 2**30:.2f} GiB)' + (' [sha256 verified]' if lfs else ' [size only]'))

print('Krea 2 assets installed. Next: cell 5 (start ComfyUI).')

Model header: 686 tensors, dtypes={'F32': 271, 'F16': 159, 'I8': 256}
Sample keys: ['model.diffusion_model.blocks.0.attn.gate.weight_scale', 'model.diffusion_model.blocks.0.attn.wk.weight_scale', 'model.diffusion_model.blocks.0.attn.wo.weight_scale']
Metadata keys: ['_quantization_metadata', 'converted_by', 'target_format']
Copying model from Drive…
Model ready: moodyKrea2MixUncensoredV8SEGoes_v80_int4.safetensors (7.02 GiB)
Downloading: qwen3vl_4b_fp8_scaled.safetensors
0.25 GiB downloaded
0.5 GiB downloaded
0.75 GiB downloaded
1.0 GiB downloaded
1.25 GiB downloaded
1.5 GiB downloaded
1.75 GiB downloaded
2.0 GiB downloaded
2.25 GiB downloaded
2.5 GiB downloaded
2.75 GiB downloaded
3.0 GiB downloaded
3.25 GiB downloaded
3.5 GiB downloaded
3.75 GiB downloaded
4.0 GiB downloaded
4.25 GiB downloaded
4.5 GiB downloaded
4.75 GiB downloaded
Ready: qwen3vl_4b_fp8_scaled.safetensors (4.88 GiB) [sha256 verified]
Downloading: qwen_image_vae.safetensors
Ready: qwen_image_vae.safetensors (0.24 GiB

In [ ]:
# 5. Start ComfyUI (local only) and verify that Krea 2 pieces are visible.
import requests

DISABLE_CUSTOM_NODES = True        # set False only if the int4 model needs a custom loader node
TEXT_ENC_FLAG = '--fp16-text-enc'  # if images come out black / NaN, try '--fp32-text-enc'
LOG = pathlib.Path('/content/comfy-t4.log')

proc = globals().get('COMFY_PROCESS')
if proc is not None and proc.poll() is None:
    print('Stopping previous ComfyUI…')
    proc.terminate()
    try:
        proc.wait(timeout=20)
    except subprocess.TimeoutExpired:
        proc.kill(); proc.wait(timeout=5)

args = [PY, 'main.py', '--listen', '127.0.0.1', '--port', '8188',
        '--disable-dynamic-vram', '--lowvram', '--fp16-unet', TEXT_ENC_FLAG, '--fp32-vae',
        '--use-pytorch-cross-attention', '--preview-method', 'none',
        '--output-directory', str(RESULTS)]
if DISABLE_CUSTOM_NODES:
    args.append('--disable-all-custom-nodes')
args.append('--enable-cors-header')   # avoids HTTP 403 when opened through the tunnel

log = open(LOG, 'w')
COMFY_PROCESS = subprocess.Popen(args, cwd=APP, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

deadline = time.monotonic() + 300
while time.monotonic() < deadline:
    if COMFY_PROCESS.poll() is not None:
        raise RuntimeError(LOG.read_text()[-8000:])
    try:
        if requests.get('http://127.0.0.1:8188/object_info', timeout=3).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError(f'ComfyUI did not become ready. See {LOG}')

def combo_options(node, field):
    try:
        spec = requests.get(f'http://127.0.0.1:8188/object_info/{node}', timeout=30).json()[node]['input']['required'][field]
        if isinstance(spec[0], list):
            return spec[0]                       # older format: [[options...], {...}]
        if len(spec) > 1 and isinstance(spec[1], dict):
            return spec[1].get('options', [])    # newer format: ['COMBO', {'options': [...]}]
    except Exception as e:
        print(f'  (could not read {node}.{field}: {e})')
    return []

unet_ok = MODEL_NAME in combo_options('UNETLoader', 'unet_name')
clip_ok = 'qwen3vl_4b_fp8_scaled.safetensors' in combo_options('CLIPLoader', 'clip_name')
vae_ok = 'qwen_image_vae.safetensors' in combo_options('VAELoader', 'vae_name')
bp_dir = APP / 'blueprints'
blueprints = [p.name for p in bp_dir.glob('*') if 'krea' in p.name.lower()] if bp_dir.is_dir() else []

print('ComfyUI is running on localhost:8188')
print('  Krea 2 model visible :', unet_ok)
print('  Text encoder visible :', clip_ok)
print('  VAE visible          :', vae_ok)
print('  Krea blueprint(s)    :', blueprints or 'NONE')
if not blueprints:
    print('HINT: this ComfyUI commit has no Krea 2 blueprint. Set COMFY_REF = "master" in cell 3, re-run cells 3-5.')
if not (unet_ok and clip_ok and vae_ok):
    print(f'HINT: something is missing from the loaders. Check the end of {LOG} and the warnings in cell 4.')

ComfyUI is running on localhost:8188
  Krea 2 model visible : True
  Text encoder visible : True
  VAE visible          : True
  Krea blueprint(s)    : ['Text to Image (Krea-2 Turbo).json', 'Text to Image (Flux.1 Krea Dev).json', 'Image Style Reference (Krea-2 Turbo).json']


In [ ]:
# 6. Open a public tunnel (Cloudflare quick tunnel) -> click the link below.
from IPython.display import display, HTML

CF = pathlib.Path('/content/cloudflared')
if not CF.exists():
    fetch('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF)
CF.chmod(0o755)

proc = globals().get('TUNNEL_PROCESS')
if proc is not None and proc.poll() is None:
    proc.terminate()
    try: proc.wait(timeout=10)
    except subprocess.TimeoutExpired: proc.kill(); proc.wait(timeout=5)

TUNNEL_LOG = pathlib.Path('/content/cloudflared.log')
tunnel_log = open(TUNNEL_LOG, 'w')
TUNNEL_PROCESS = subprocess.Popen([str(CF), 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
                                  stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)

url = None
deadline = time.monotonic() + 90
while time.monotonic() < deadline and url is None:
    if TUNNEL_PROCESS.poll() is not None:
        raise RuntimeError(TUNNEL_LOG.read_text()[-4000:])
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', TUNNEL_LOG.read_text())
    if m:
        url = m.group(0)
    else:
        time.sleep(1)
if url is None:
    raise TimeoutError(f'No tunnel URL after 90s. See {TUNNEL_LOG}')

display(HTML(f'<h3>ComfyUI: <a href="{url}" target="_blank">{url}</a></h3>'
             '<p>เปิดลิงก์ในแท็บใหม่ (ถ้าขึ้นว่าหาโดเมนไม่เจอ รอ 10–20 วินาทีแล้วรีเฟรช). '
             'ลิงก์นี้เป็นสาธารณะขณะที่ยังรันอยู่ — อย่าแชร์ และหยุดด้วย Runtime → Disconnect and delete runtime เมื่อเสร็จ</p>'))

Downloading: cloudflared
